<a href="https://colab.research.google.com/github/Kevin911212/CLIP-project/blob/main/CLIP_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install python-dotenv chromadb open-clip-torch
!pip install -U datasets
!pip install -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.6/201.6 kB 19.7 MB/s eta 

In [ ]:
from huggingface_hub import login
login("")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
import os
import requests
import torch
from transformers import CLIPProcessor, CLIPModel, GPT2LMHeadModel, GPT2Tokenizer
from chromadb.utils.embedding_functions import OpenCLIPEmbeddingFunction
from chromadb import PersistentClient
from PIL import Image

# 初始化 CLIP 模型
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


# 定義自訂的嵌入函數類
class CustomCLIPEmbeddingFunction:
    def __call__(self, input):
        if isinstance(input, list):
            inputs = clip_processor(text=input, return_tensors="pt", padding=True)
            with torch.no_grad():
                embeddings = clip_model.get_text_features(**inputs)
            return embeddings.cpu().numpy()
        else:
            raise ValueError("Input to the embedding function must be a list of strings.")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
import os
from PIL import Image
import warnings
import numpy as np
from datasets import load_dataset
import chromadb
from chromadb.utils.embedding_functions import OpenCLIPEmbeddingFunction
from chromadb.utils.data_loaders import ImageLoader

# Suppress warnings
warnings.filterwarnings("ignore")

# 設定資料夾
dataset_folder = "./dataset/fashion"
os.makedirs(dataset_folder, exist_ok=True)

# 下載資料集
print("Downloading the dataset...")
ds = load_dataset("jinaai/fashion-captions-de")


# 儲存資料集圖片到本地
def save_images(dataset, dataset_folder, num_images=500):
    os.makedirs(dataset_folder, exist_ok=True)
    for i in range(num_images):
        try:
            print(f"Saving image {i+1} of {num_images}")
            image = dataset["train"][i]["image"]
            image.save(os.path.join(dataset_folder, f"fashion_{i+1}.png"))
        except Exception as e:
            print(f"Error saving image {i+1}: {e}")
    print(f"Saved {num_images} images to {dataset_folder}")

save_images(ds, dataset_folder, num_images=500)

# 初始化 ChromaDB
print("Initializing ChromaDB...")
chroma_client = chromadb.PersistentClient(path="./data/fashion.db")
embedding_function = OpenCLIPEmbeddingFunction()
image_loader = ImageLoader()

fashion_collection = chroma_client.get_or_create_collection(
    "fashion_collection",
    embedding_function=embedding_function,
    data_loader=image_loader,
)

# 添加圖片和描述到 ChromaDB
print("Adding images and descriptions to ChromaDB...")
ids = []
uris = []
metadatas = []

image_files = sorted([f for f in os.listdir(dataset_folder) if f.endswith(".png")])
for i, filename in enumerate(image_files):
    file_path = os.path.join(dataset_folder, filename)
    original_description = ds["train"][i]["text"]  # 從資料集中獲取描述文字
    ids.append(str(i))
    uris.append(file_path)
    metadatas.append({"description": original_description})

if ids and uris:
    fashion_collection.add(ids=ids, uris=uris, metadatas=metadatas)
    print(f"Added {len(ids)} images and descriptions to the database in Traditional Chinese.")
else:
    print("No images found to add to the database.")

# 驗證資料庫是否有內容
print(f"Total images in the collection: {fashion_collection.count()}")

README.md: 0.00B [00:00, ?B/s]

(…)-00000-of-00001-4c3f5333e9df536a.parquet:   0%|          | 0.00/267M [00:00<?, ?B/s]

(…)-00000-of-00001-eef66a2cc89fefb0.parquet:   0%|          | 0.00/53.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2001 [00:00<?, ? examples/s]

Saving image 1 of 500
Saving image 2 of 500
Saving image 3 of 500
Saving image 4 of 500
Saving image 5 of 500
Saving image 6 of 500
Saving image 7 of 500
Saving image 8 of 500
Saving image 9 of 500
Saving image 10 of 500
Saving image 11 of 500
Saving image 12 of 500
Saving image 13 of 500
Saving image 14 of 500
Saving image 15 of 500
Saving image 16 of 500
Saving image 17 of 500
Saving image 18 of 500
Saving image 19 of 500
Saving image 20 of 500
Saving image 21 of 500
Saving image 22 of 500
Saving image 23 of 500
Saving image 24 of 500
Saving image 25 of 500
Saving image 26 of 500
Saving image 27 of 500
Saving image 28 of 500
Saving image 29 of 500
Saving image 30 of 500
Saving image 31 of 500
Saving image 32 of 500
Saving image 33 of 500
Saving image 34 of 500
Saving image 35 of 500
Saving image 36 of 500
Saving image 37 of 500
Saving image 38 of 500
Saving image 39 of 500
Saving image 40 of 500
Saving image 41 of 500
Saving image 42 of 500
Saving image 43 of 500
Saving image 44 of 5

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Adding images and descriptions to ChromaDB...
Added 500 images and descriptions to the database in Traditional Chinese.
Total images in the collection: 500


In [ ]:
from matplotlib import pyplot as plt
# 查詢資料庫 (文字查圖片)
def query_db(query, results=3):
    print(f"Querying the database for: {query}")
    try:
        query_results = fashion_collection.query(
            query_texts=[query],
            n_results=results,
            include=["uris", "metadatas"]
        )
        # 將結果組成 (圖片路徑, 描述)
        gallery_items = []
        for uri, meta in zip(query_results["uris"][0], query_results["metadatas"][0]):
            caption = meta.get("description", "No description")
            gallery_items.append((uri, caption))
        return gallery_items
    except Exception as e:
        print(f"Error while querying the database: {e}")
        return []


def show_image_from_uri(uri):
    img = Image.open(uri)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

# 顯示查詢結果
def print_results(results):
    if not results or "uris" not in results or not results["uris"]:
        print("No results found.")
        return

    print("\nHere are the closest matching images:\n")
    for idx, uri in enumerate(results["uris"][0]):
        print(f"Result {idx + 1}:")
        print(f"Image Path: {uri}")
        print(f"Distance: {results['distances'][0][idx]}")
        show_image_from_uri(uri)
        print("\n")

In [ ]:
def image_to_image_search(image, results=3):
    print("Searching for similar images...")
    try:
        # 使用 CLIP 將圖片轉成嵌入
        image = image.convert("RGB").resize((224, 224))
        img_array = np.array(image)
        embedding = embedding_function([img_array])

        # 查詢最相似的圖片
        query_results = fashion_collection.query(
            query_embeddings=embedding,
            n_results=results,
            include=["uris", "metadatas"]
        )
        gallery_items = []
        for uri, meta in zip(query_results["uris"][0], query_results["metadatas"][0]):
            caption = meta.get("description", "No description")
            gallery_items.append((uri, caption))
        return gallery_items
    except Exception as e:
        print(f"Error processing the image: {e}")
        return []

In [ ]:
def image_to_text_search(image, results=3):
    try:
        image = image.convert("RGB").resize((224, 224))
        img_array = np.array(image)
        embedding = embedding_function([img_array])

        results = fashion_collection.query(
            query_embeddings=embedding,
            n_results=results,
            include=["metadatas", "distances"]
        )
        descriptions = [
            f"{i+1}. {meta['description']}"
            for i, meta in enumerate(results["metadatas"][0])
        ]
        return "\n\n".join(descriptions)
    except Exception as e:
        return f"發生錯誤：{str(e)}"

In [ ]:
import torch
from transformers import pipeline

# 建立 text-generation pipeline
llm_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

# 對話函數：自動串接歷史訊息並用 template 包起來
def fashion_chat(user_input, history):
    messages = [{"role": "system", "content": "You are a helpful and friendly fashion assistant."}]

    # 加入歷史訊息
    for user, assistant in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": assistant})

    messages.append({"role": "user", "content": user_input})

    # 使用 tokenizer 套用聊天模板
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = llm_pipeline(prompt, max_new_tokens=200, do_sample=False)
    reply = outputs[0]["generated_text"].split(prompt)[-1].strip()
    return reply

Device set to use cuda:0


In [ ]:
def generate_query_from_gpt(user_input):
    prompt = f"You are a helpful fashion assistant. Convert the following request into a short and clear English search query.\nUser: \"{user_input}\"\nQuery:"
    output = llm_pipeline(prompt, max_new_tokens=30, do_sample=False)
    query = output[0]["generated_text"].split("Query:")[-1].strip()
    return query

In [ ]:
def query(query, results=5):
    try:
        result = fashion_collection.query(
            query_texts=[query],
            n_results=results,
            include=["uris", "distances"]
        )
        return result["uris"][0] if "uris" in result else []
    except Exception as e:
        print(f"Query error: {e}")
        return []

In [ ]:
# === Chatbot 回應 + 圖像檢索 ===
chat_history = []

def chatbot_interface(message, history):
    history.append((message, None))

    try:
        # GPT 生成搜尋詞
        search_query = generate_query_from_gpt(message)

        # 查詢圖像資料庫（CLIP）
        image_uris = query_db(search_query, results=3)

        # 回應內容
        reply = f"🔍 Based on your request, I searched for: **{search_query}**\nHere are some styles you might like:"
        history[-1] = (message, reply)

        return history, image_uris, ""
    except Exception as e:
        history[-1] = (message, f"Error: {str(e)}")
        return history, []

In [ ]:
import gradio as gr

# === Gradio UI ===
with gr.Blocks(title="Fashion CLIP + GPT Chat Assistant") as demo:
    gr.Markdown("## Fashion Search Assistant\nFind your style using AI + Visual Search")

    # Text → Image
    with gr.Tab("Text → Image"):
        text_input = gr.Textbox(label="Describe the style (e.g. 'a red summer dress')")
        topk_slider = gr.Slider(1, 10, value=5, step=1, label="Number of Results")
        text_output = gr.Gallery(label="Matching Items", columns=3)
        text_button = gr.Button("Search")
        text_button.click(fn=lambda q, k: query_db(q, results=k),
                          inputs=[text_input, topk_slider],
                          outputs=text_output)

    # Image → Image
    with gr.Tab("Image → Image"):
        img_input = gr.Image(label="Upload an image you like", type="pil")
        topk_slider2 = gr.Slider(1, 10, value=5, step=1, label="Similar Images")
        img_output = gr.Gallery(label="Similar Fashion", columns=3)
        img_button = gr.Button("Find Similar")
        img_button.click(fn=image_to_image_search,
                         inputs=[img_input, topk_slider2],
                         outputs=img_output)

    # Image → Description
    with gr.Tab("Image → Description"):
        img_input_text = gr.Image(label="Upload a fashion image", type="pil")
        topk_slider3 = gr.Slider(1, 10, value=3, step=1, label="Number of Descriptions")
        text_result = gr.Textbox(label="Most Relevant Descriptions", lines=6)
        img_button_text = gr.Button("Generate Description")
        img_button_text.click(fn=image_to_text_search,
                              inputs=[img_input_text, topk_slider3],
                              outputs=text_result)

    #chatbot = gr.Chatbot(label="Assistant")
    with gr.Tab("AI assistant"):
        gr.Markdown("""
        ** Hello! I'm your Fashion AI Assistant.**
        Tell me what kind of clothing you're looking for, and I'll recommend matching images.
        You can say things like:

        - *"I want a red summer dress."*
        - *"Show me outfits suitable for winter."*
        - *"Find something elegant for a wedding."*

        Let's find your style!
        """)

        chatbot = gr.Chatbot(label="Assistant")
        txt_input = gr.Textbox(placeholder="Describe the style you want...", label="Your Fashion Query")
        send_btn = gr.Button("Send")
        clear_btn = gr.Button("Clear Chat")
        result_gallery = gr.Gallery(label="Recommended Items", columns=3, height=500)

        send_btn.click(
          chatbot_interface,
          inputs=[txt_input, chatbot],
          outputs=[chatbot, result_gallery, txt_input]
        )
        txt_input.submit(
          chatbot_interface,
          inputs=[txt_input, chatbot],
          outputs=[chatbot, result_gallery, txt_input]
        )

        def clear_chat():
            return [], []

        clear_btn.click(clear_chat, outputs=[chatbot, result_gallery])

# 啟動介面
demo.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fcd04ccf9baf23a622.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
